# Kaczmarz Preconditioning for CG
We will finally consider using a randomized kaczmarz method as a preconditioner for CG. To do this we will first load in the required packages and prewritten functions.

In [1]:
using RandLinearAlgebra
using LinearAlgebra
using Random
include("helpers/cg.jl")

pcg

The `pcg` function has the following signature
```
function pcg(
    A, 
    b; 
    x = zeros(size(b, 1)), 
    L = IdentityPreconditioner(), 
    maxit = size(b, 1), 
    threshold = 1e-12
)
```
where `L` is a left preconditioner which will be applied using the `ldiv!` function. So to define a new preconditioner we need to create a new data structure and a new `ldiv!` function. As an example, here is the `IdentityPreconditioner`,
```
mutable struct IdentityPreconditioner end

function ldiv!(x::AbstractVector, L::IdentityPreconditioner, y::AbstractVector)
    x .= y
end
```
 To use Kaczmarz as a preconditioner, we will first define our data structure.

In [2]:
mutable struct KaczmarzPreconditioner  <: Preconditioner
    A::AbstractMatrix
    x::AbstractVector
    recipe::KaczmarzRecipe
end

You will notice that this structure contains a recipe rather than ingredient structure. So make this easy to use we will also define a constructor that allows us to specify our `Compressor` for use in Kaczmarz and the maximum number of iterations of Kaczmarz we want to run.

In [3]:
function KaczmarzPreconditioner(
    A;
    x = zeros(size(A, 1)),
    maxit = size(A, 1),
    comp = Sampling(
        compression_dim = 2,
        distribution = Uniform()
    )
)
    # Form the Kaczmarz recipe
    recipe = complete_solver(
        Kaczmarz(compressor = comp, log = BasicLogger(max_it = maxit), alpha = 0.75),
        x,
        A,
        x
    )
    return KaczmarzPreconditioner(A, x, recipe)
end

KaczmarzPreconditioner

Finally, we can define our `ldiv!` function for the Kaczmarz solver. This requires us to wrap the `rsolve!` function.

In [4]:
function ldiv!(x::AbstractVector, L::KaczmarzPreconditioner, y::AbstractVector)
    rsolve!(L.recipe, x, L.A, y)
end

ldiv! (generic function with 2 methods)

Finally we can test the preconditioner with Strohmer Vershynin sampling.

In [5]:
Random.seed!(12312)
A = rand(100,80);
A = A' * A;
xsol = rand(80);
b = A * xsol;
xu = pcg(A, b, maxit = 300, threshold = 1e-10)
print("Unpreconditioned took") 
printstyled(" $(xu[2]) iterations ", bold = true, color = :yellow) 
print("with a final error of $(norm(xu[1] - xsol))")



Unpreconditioned took 88 iterations with a final error of 1.420553606442962e-7

In [7]:
# define the preconditioner with Strohmer Vershynin
precon = KaczmarzPreconditioner(A, comp = Sampling(compression_dim = 70, distribution = L2Norm()))
x = pcg(A, b, L = precon, maxit = 300, threshold = 1e-10)
print("SV Preconditioned took") 
printstyled(" $(x[2]) iterations ", bold = true, color = :yellow) 
print("with a final error of $(norm(x[1] - xsol))")

SV Preconditioned took 57 iterations with a final error of 4.584920765412739e-7

Now test with Uniform Sampling.

In [8]:
precon = KaczmarzPreconditioner(A, comp = Sampling(compression_dim = 70, distribution = Uniform()))

x = pcg(A, b, L = precon, maxit = 300, threshold = 1e-10)
print("Uniform preconditioned took")
printstyled(" $(x[2]) iterations ", bold = true, color = :yellow) 
print("with a final error of $(norm(x[1] - xsol))")

Uniform preconditioned took 23 iterations with a final error of 3.9525726180662844e-7

Test with Gaussian compression.

In [9]:
precon = KaczmarzPreconditioner(A, comp = Gaussian(compression_dim = 70))

x = pcg(A, b, L = precon, maxit = 300, threshold = 1e-10)
print("Gaussian preconditioned took")
printstyled(" $(x[2]) iterations ", bold = true, color = :yellow) 
print("with a final error of $(norm(x[1] - xsol))")

Gaussian preconditioned took 44 iterations with a final error of 3.8078104580662707e-7